# 🛠️ Setup
---

## This notebook draws from the original paragraph by  ERTUĞRUL DEMIR
- **Notebook**
  - Kaggle URL: https://www.kaggle.com/code/datafan07/train-your-own-tokenizer/notebook

In [ ]:
pwd

In [ ]:
cd /kaggle/input/readabili

In [ ]:
!pip install py_readability_metrics-1.4.5-py3-none-any.whl

In [ ]:
cd /kaggle/working

In [ ]:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import torch
# import numpy as np

# model = AutoModelForSequenceClassification.from_pretrained('vectara/hallucination_evaluation_model')
# tokenizer = AutoTokenizer.from_pretrained('vectara/hallucination_evaluation_model')
# model.to("cuda")

In [ ]:
# def evalute_hallu(text):

#     inputs = tokenizer(text, return_tensors='pt', padding=True, truncation = True).to("cuda")

#     model.eval()
#     scores = []
#     with torch.no_grad():
#         outputs = model(**inputs)
#         logits = outputs.logits.cpu().detach().numpy()
#         # convert logits to probabilities
#         score = 1 / (1 + np.exp(-logits)).flatten()
#     return score[0]

In [ ]:
import sys
import gc

import pandas as pd
from sklearn.model_selection import StratifiedKFold
import numpy as np
from sklearn.metrics import roc_auc_score
import numpy as np
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer,
)

from datasets import Dataset
from tqdm.auto import tqdm
from transformers import PreTrainedTokenizerFast

from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import VotingClassifier
from readability import Readability
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow.keras.metrics import AUC, Recall, Precision
roc_auc = AUC(curve='ROC')

def extract_scores(text):
    try:
        r = Readability(text)
        scores =  [r.flesch_kincaid().score, r.flesch().score, r.gunning_fog().score, r.coleman_liau().score, r.dale_chall().score, r.ari().score, r.linsear_write().score, r.spache().score]
        return [round(x, 3) + 1000 for x in scores]
    except:
        return [0]*8

In [ ]:
submit = True

In [ ]:
test = pd.read_csv('/kaggle/input/llm-detect-ai-generated-text/test_essays.csv')
sub = pd.read_csv('/kaggle/input/llm-detect-ai-generated-text/sample_submission.csv')
org_train = pd.read_csv('/kaggle/input/llm-detect-ai-generated-text/train_essays.csv')
train = pd.read_csv("/kaggle/input/daigt-v2-train-dataset/train_v2_drcat_02.csv", sep=',')

In [ ]:
train = train.drop_duplicates(subset=['text'])
train.reset_index(drop=True, inplace=True)
# train = train.sample(10_00, random_state = 1)

In [ ]:
# train['scores'] = train['text'].map(extract_scores)
# scores_train = train['scores'].apply(pd.Series)
# scores_train.columns = [ f"scores_{x}" for x in scores_train.columns]
# gc.collect()

# # scores_train.to_csv("scores_train.csv", index = False)

In [ ]:
# scores_train = pd.read_csv("/kaggle/input/scorest/scores_train.csv")

In [ ]:
# test['scores'] = test['text'].map(extract_scores)
# scores_test = test['scores'].apply(pd.Series)
# scores_test.columns = [ f"scores_{x}" for x in scores_test.columns]
# gc.collect()

In [ ]:
LOWERCASE = False
VOCAB_SIZE = 30522

In [ ]:
### dziele na train test dla lstm
df_train, df_test = train_test_split(train, test_size = 0.3, random_state = 42)

In [ ]:
# Creating Byte-Pair Encoding tokenizer
raw_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
raw_tokenizer.normalizer = normalizers.Sequence([normalizers.NFC()] + [normalizers.Lowercase()] if LOWERCASE else [])
raw_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
special_tokens = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=special_tokens)

#### tutaj podmianka na test

if submit:
    dataset = Dataset.from_pandas(test[['text']])
else:
    dataset = Dataset.from_pandas(train[['text']])
    

def train_corp_iter(): 
    for i in range(0, len(dataset), 1000):
        yield dataset[i : i + 1000]["text"]
raw_tokenizer.train_from_iterator(train_corp_iter(), trainer=trainer)
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=raw_tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
)

tokenized_texts_test = []
for text in tqdm(test['text'].tolist()):
    tokenized_texts_test.append(tokenizer.tokenize(text))

tokenized_texts_train = []
for text in tqdm(train['text'].tolist()):
    tokenized_texts_train.append(tokenizer.tokenize(text))

    
tokenized_texts_train_lstm = []
for text in tqdm(df_train['text'].tolist()):
    tokenized_texts_train_lstm.append(tokenizer.tokenize(text))
    
    
tokenized_texts_test_lstm = []
for text in tqdm(df_test['text'].tolist()):
    tokenized_texts_test_lstm.append(tokenizer.tokenize(text))

In [ ]:
def dummy(text):
    return text
vectorizer = TfidfVectorizer(ngram_range=(3, 5), lowercase=False, sublinear_tf=True, analyzer = 'word',
    tokenizer = dummy,
    preprocessor = dummy,
    token_pattern = None, strip_accents='unicode')


if submit:
    vectorizer.fit(tokenized_texts_test)
else:
    vectorizer.fit(tokenized_texts_train)

# Getting vocab
vocab = vectorizer.vocabulary_

# print(vocab)

vectorizer = TfidfVectorizer(ngram_range=(3, 5), lowercase=False, sublinear_tf=True, vocabulary=vocab,
                            analyzer = 'word',
                            tokenizer = dummy,
                            preprocessor = dummy,
                            token_pattern = None, strip_accents='unicode'
                            )

tf_train = vectorizer.fit_transform(tokenized_texts_train)
tf_test = vectorizer.transform(tokenized_texts_test)


tf_train_lstm = vectorizer.fit_transform(tokenized_texts_train_lstm)
tf_test_lstm = vectorizer.transform(tokenized_texts_test_lstm)

del vectorizer
gc.collect()

In [ ]:
from keras.models import Sequential
from keras.layers import LSTM, Dense
from keras.utils import Sequence

In [ ]:
class SparseDataGenerator(Sequence):
    def __init__(self, x_set, y_set, batch_size):
        self.x, self.y = x_set, y_set
        self.batch_size = batch_size
        self.indices = np.arange(self.x.shape[0])

    def __len__(self):
        # Zmiana sposobu obliczania długości generatora
        return int(np.ceil(self.x.shape[0] / float(self.batch_size)))

    def __getitem__(self, idx):
        inds = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_x = self.x[inds].toarray()
        batch_y = self.y[inds]

        # Reshape dla LSTM
        batch_x = np.reshape(batch_x, (batch_x.shape[0], 1, batch_x.shape[1]))

        return batch_x, batch_y

In [ ]:
batch_size = 4
train_generator = SparseDataGenerator(tf_train_lstm, df_train['label'].values, batch_size)
test_generator = SparseDataGenerator(tf_test_lstm, df_test['label'].values, batch_size)

test_generator_sub = SparseDataGenerator(tf_test,np.array([0]*tf_test.shape[0]), batch_size)

In [ ]:
n_features = tf_train.shape[1]


model = Sequential()
model.add(LSTM(units=50, activation='relu', input_shape=(1, n_features)))
model.add(Dense(units=1, activation='sigmoid'))


model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', roc_auc, Recall(), Precision()])

In [ ]:
# Trenowanie modelu
model.fit(train_generator, epochs=2, validation_data=test_generator)

In [ ]:
sub['generated'] = model.predict(test_generator_sub)

In [ ]:
sub.to_csv('submission.csv', index=False)

In [ ]:
# from scipy.sparse import csr_matrix, hstack

# df_sparse = csr_matrix(scores_train.values)
# tf_train = hstack([df_sparse, tf_train])

# df_sparse = csr_matrix(scores_test.values)
# tf_test = hstack([df_sparse, tf_test])
# gc.collect()

In [ ]:
# y_train = train['label'].values

In [ ]:
# if len(test.text.values) <= 5:
#     sub.to_csv('submission.csv', index=False)
# else:
#     clf = MultinomialNB(alpha=0.02)
# #     clf2 = MultinomialNB(alpha=0.01)
#     sgd_model = SGDClassifier(max_iter=8000, tol=1e-4, loss="modified_huber") 
#     p6={'n_iter': 1500,'verbose': -1,'objective': 'cross_entropy','metric': 'auc',
#         'learning_rate': 0.05073909898961407, 'colsample_bytree': 0.726023996436955,
#         'colsample_bynode': 0.5803681307354022, 'lambda_l1': 8.562963348932286, 
#         'lambda_l2': 4.893256185259296, 'min_data_in_leaf': 115, 'max_depth': 23, 'max_bin': 898}
#     lgb=LGBMClassifier(**p6)
#     cat=CatBoostClassifier(iterations=1000,
#                            verbose=0,
#                            l2_leaf_reg=6.6591278779517808,
#                            learning_rate=0.005689066836106983,
#                            allow_const_label=True,loss_function = 'CrossEntropy')
#     weights = [0.07,0.31,0.31,0.31]
 
#     ensemble = VotingClassifier(estimators=[('mnb',clf),
#                                             ('sgd', sgd_model),
#                                             ('lgb',lgb), 
#                                             ('cat', cat)
#                                            ],
#                                 weights=weights, voting='soft', n_jobs=-1)
#     ensemble.fit(tf_train, y_train)
#     gc.collect()
#     final_preds = ensemble.predict_proba(tf_test)[:,1]
#     final_preds = [1 if x > 0.8 else x for x in final_preds]
#     sub['generated'] = final_preds
#     sub.to_csv('submission.csv', index=False)
#     sub